In [ ]:
from pathlib import Path

from ultralytics import YOLO

from libs.dataset import (
    anonymize_dataset,
    create_dataset,
    create_retinal_segmentation_testset,
    download_images,
)
from libs.plotting import plot_LayerCAM, plot_LayerCAM_image
from libs.train import predict_segmentations_folder, test, train
from libs.utils import find_root_repository

We performed a KFOLD on the dataset to evaluate the YOLO model. Now we just train the model on all the dataset.

In [ ]:
root = find_root_repository()
train_csv_path = str(root / "dataset/source/csv/sources_kfold_train.csv")
test_csv_path = str(root / "dataset/source/csv/sources_kfold_test.csv")
images_ref_path = str(root / "dataset/image_ref")

anonymize_dataset(train_csv_path)
download_images(train_csv_path, f"{images_ref_path}/train")

anonymize_dataset(test_csv_path)
download_images(test_csv_path, f"{images_ref_path}/test")

## Dataset

IF YOU NEVER DOWNLOAD LOCALY IMAGES + ANONYMISE IT YOU TO

In [ ]:
train_csv_path = str(find_root_repository()/ "dataset/source/csv/sources_kfold_train.csv")
test_csv_path = str(find_root_repository()/ "dataset/source/csv/sources_kfold_test.csv")
output_train = str(find_root_repository()/ "dataset/main/train")
output_test = str(find_root_repository()/ "dataset/main/test")

create_dataset(train_csv_path, output_train, source_col="LOCAL_REF")
create_dataset(test_csv_path, output_test, source_col="LOCAL_REF")

## Training and Test 

Train on all the dataset and evaluate (test) on new image. We don't need validation dataset because we already have found best parameters

In [ ]:
config = {"model": "../models/yolo11n-cls.pt", 
            "optimizer": 'SGD',
            "batch": 16,
            "epochs": 100,
            "imgsz": 512}

dataset_path = str(find_root_repository() / "dataset/main")

model = train("eyeSideClassificationV1_nano", dataset_path, config)
test(model, dataset_path)

## Error tests

In [ ]:
# model.predict("moment0ff.png", visualize=True)

r = model.predict("../dataset/main/test/left/0082.png", imgsz=512, device=0, verbose=False, visualize=True)[0]
pred = model.names[r.probs.top1]

print(f"pref: {pred}, conf: {r.probs.top1conf.item()}")

In [ ]:
def print_error(model, dataset_test: str, config: dict):
    
    def _predict_class(class_name):
        path = Path(dataset_test, class_name)
        
        if not path.exists():
            print(f"Class {class_name} doesn't exist ! {path}")
            return False
        
        results = model(path, imgsz=config["imgsz"], stream=True, verbose=False)
        for r in results:
            if model.names[r.probs.top1] != class_name:
                print(f"Failed, predicted: {model.names[r.probs.top1]}, ground truth: {class_name} (conf: {r.probs.top1conf}) (path: {r.path})")

        return True
    
    _predict_class("left")
    _predict_class("right")

config = {"model": "../models/yolo11n-cls.pt", 
            "optimizer": 'SGD',
            "batch": 16,
            "epochs": 100,
            "imgsz": 512}


model = YOLO(find_root_repository() / "other/weights/best.pt")

print_error(model, str(find_root_repository() / "dataset/main/test"), config)

## XIA

### LayerCAM
Some thing that could be interesting is to view the layer-CAM to see the region that model look at to take a decision

In [ ]:
model = YOLO(find_root_repository() / "other/weights/LateralityCLS.pt")

In [ ]:
plot_LayerCAM(model, str(find_root_repository() / "dataset/main/test/left"), "left")

In [ ]:
plot_LayerCAM(model, str(find_root_repository() / "dataset/main/test/left"), "right")

Code for the article

In [ ]:
model = YOLO(find_root_repository() / "other/weights/best.pt")
paths = [("left/0000.png" , "left" ),
         ("right/0009.png", "right"),
         ("left/0022.png" , "left" ),
         ("right/0013.png", "right"),
         ("right/0021.png", "right"),
         ("left/0042.png" , "left" ),
         ("right/0051.png", "right"),
         ("left/0074.png" , "left" )]

for img, gd in paths:
    plot_LayerCAM_image(model, str(find_root_repository() / f"dataset/main/test/{img}"), gd)

In [ ]:
model = YOLO(find_root_repository() / "other/weights/best.pt")

# 13
# 34
# 42

L = [
    "0013.png",
    "0034.png",
    "0042.png"
]

for img_file in L:
    img_path = find_root_repository() / "dataset" / "image_ref" / "test" / img_file
    plot_LayerCAM_image(model, img_path, "")

### Experiment: Is model detect retinal's vessel and can based it's prediction on it

In [ ]:
create_retinal_segmentation_testset("dataset/source/txt/retina_measures.txt", 
      "dataset/source/csv/sources_kfold_train.csv",
      "dataset/testseg",
      "dataset/source/csv/testseg.csv")

In [ ]:
model = YOLO("weights/eyeSideClassification_nano.pt")
predict_segmentations_folder(model, "dataset/testseg", "dataset/source/csv/testseg.csv")

In [ ]:
model = YOLO("weights/eyeSideClassification.pt")
plot_LayerCAM_image(model, "dataset/testseg/0000.png", "left")
plot_LayerCAM_image(model, "dataset/moment0ff.png", "")

#### Result:
- accuracy artery: 0.57
- accuracy vein: 0.69
- accuracy both on the same image: 0.63

We can't conclude that the classifier detect or can based its predictions 
on the vessel's retinian.

It's just prove that the classifier need other things like choroidian vessels
or optic disk information for a better accuracy

## Export

In [ ]:
# export to MATLAB ONNX (opset version 17 important !)
model = YOLO("../other/weights/LateralityClassification.pt")
model.export(format="onnx", opset=17)